# 10 — Đóng gói HCO–VNLS + LightGBM cutoff 75%

Khóa mask 17 cột từ notebook 05b và 09; fit lại cùng pipeline trên train + validation để xuất model cho app. Điểm validation trong báo cáo là điểm của model fit trên train, **không phải điểm mới của artifact đã fit lại**. Không dùng test để chọn hoặc fit model.


In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import sys
import joblib
import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from IPython.display import display
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'research_code').is_dir() and (path / 'shared_code').is_dir()), None)
assert ROOT is not None, 'Chạy từ thư mục dự án hoặc notebooks.'
sys.path.insert(0, str(ROOT))
from shared_code.thuc_nghiem import NGHIEN_CUU, KET_QUA, nap_du_lieu, chon_dac_trung
from research_code.models import ModelConfig, make_pipeline
CUTOFF, SEED_TIM_MASK, NGUONG = .75, 41, .5
CONFIG = ModelConfig('LightGBM_search_80', LGBMClassifier(
    n_estimators=80, learning_rate=.055, num_leaves=15,
    min_child_samples=35, verbosity=-1, random_state=41, n_jobs=4))
THU_MUC_MODEL = NGHIEN_CUU / 'mo_hinh'
THU_MUC_MODEL.mkdir(exist_ok=True)
DUONG_DAN_MODEL = THU_MUC_MODEL / 'model_hco_vnls_lightgbm_cutoff75.joblib'
DUONG_DAN_METADATA = THU_MUC_MODEL / 'THONG_TIN_MODEL_HCO_VNLS_LIGHTGBM_75.json'
print('Nguồn:', NGHIEN_CUU)


## 1. Khóa đúng mask và dữ liệu trước khi fit

Đối chiếu hai bảng 05b và 09 để tránh xuất nhầm mask; kiểm tra cột đầu vào, thứ tự cột và các nhóm sinh viên giữa train, validation, test. `risk`, `split` và `id_student` không được là feature.


In [2]:
reference = pd.read_csv(KET_QUA / 'hco_vnls_lightgbm_validation.csv')
reference_row = reference.loc[(reference.cutoff == CUTOFF) & (reference.method == 'HCO-VNLS')].iloc[0]
confirmation = pd.read_csv(KET_QUA / 'kiem_chung_mask_validation.csv')
confirmed_row = confirmation.loc[(confirmation.cutoff == CUTOFF) &
    (confirmation.method == 'HCO-VNLS') & (confirmation.seed == SEED_TIM_MASK)].iloc[0]
selected = reference_row.selected_columns.split('|')
assert selected == confirmed_row.selected_columns.split('|')
assert len(selected) == 17 and len(set(selected)) == 17
frame = nap_du_lieu(CUTOFF)
assert set(frame.split) == {'train', 'validation', 'test'}
assert set(selected).issubset(chon_dac_trung(frame))
assert not set(selected).intersection({'id_student', 'risk', 'split', 'final_result'})
groups = {part: set(frame.loc[frame.split == part, 'id_student'])
          for part in ('train', 'validation', 'test')}
assert all(not groups[left].intersection(groups[right])
           for left, right in (('train', 'validation'), ('train', 'test'), ('validation', 'test')))
fit_rows = frame.loc[frame.split.isin(['train', 'validation'])]
test_rows = frame.loc[frame.split == 'test']
print('17 cột:', ', '.join(selected))
display(pd.DataFrame({'tập': frame.groupby('split').size().index,
                      'số dòng': frame.groupby('split').size().values}))
print('AP validation tham chiếu (fit train):', round(reference_row.average_precision_risk, 6))


17 cột: code_module, gender, highest_education, age_band, studied_credits, date_registration, total_clicks_to_t, clicks_resource_to_t, clicks_url_to_t, active_days_to_t, assessments_submitted_to_t, late_submissions_to_t, active_day_ratio, submission_rate, days_since_last_activity, clicks_recent_7, clicks_recent_30


,tập,số dòng
0,test,3447
1,train,16297
2,validation,3415


AP validation tham chiếu (fit train): 0.896757


## 2. Fit pipeline sẽ được mang vào app

Dùng đúng cấu hình LightGBM 80 cây và `make_pipeline` của notebook 05b. Bộ điền thiếu và one-hot nằm trong file model; thống kê tiền xử lý chỉ học từ train + validation. Test không tham gia fit.


In [3]:
pipeline = make_pipeline(fit_rows[selected], CONFIG)
pipeline.fit(fit_rows[selected], fit_rows.risk)
assert list(pipeline.named_steps['model'].classes_) == [0, 1]
print('Đã fit trên', len(fit_rows), 'dòng train + validation; test không dùng để fit.')


Đã fit trên 19712 dòng train + validation; test không dùng để fit.


## 3. Lưu và kiểm tra cục model

App cần tạo 17 đặc trưng từ dữ liệu **trước cutoff 75%**, truyền đúng schema vào pipeline; bảy bảng thô không phải đầu vào trực tiếp. Ngưỡng 0,5 dùng để minh họa, chưa hiệu chỉnh cho vận hành. Model này không áp dụng ở cutoff 50%.


In [ ]:
schema = [{'name': column, 'kind': 'category' if fit_rows[column].dtype == 'object' else 'number'}
          for column in selected]
metadata = {
    'model_name': 'HCO-VNLS + LightGBM 80 cây — cutoff 75%',
    'cutoff_fraction': CUTOFF, 'optimizer_seed': SEED_TIM_MASK,
    'model_random_state': 41, 'feature_columns': selected, 'input_schema': schema,
    'positive_label': 1, 'threshold': NGUONG,
    'training_policy': 'Fit train + validation sau khi khóa mask; không dùng test để fit.',
    'training_rows': len(fit_rows), 'test_rows_excluded': len(test_rows),
    'validation_reference_train_only': {
        'average_precision_risk': float(reference_row.average_precision_risk),
        'accuracy': float(reference_row.accuracy),
        'recall_risk': float(reference_row.recall_risk)},
    'mask_source': '05b_hco_vnls_lightgbm.ipynb; xác nhận bởi 09_kiem_chung_dac_trung_hco_vnls.ipynb',
    'feature_data_source': 'results/du_lieu_cutoff_75.csv',
    'data_sha256': sha256((KET_QUA / 'du_lieu_cutoff_75.csv').read_bytes()).hexdigest(),
    'versions': {'python': sys.version.split()[0], 'scikit_learn': sklearn.__version__,
                 'lightgbm': lightgbm.__version__, 'pandas': pd.__version__,
                 'joblib': joblib.__version__},
}
artifact = {'pipeline': pipeline, 'feature_columns': selected, 'threshold': NGUONG,
            'cutoff_fraction': CUTOFF, 'metadata': metadata}
joblib.dump(artifact, DUONG_DAN_MODEL)
metadata['model_sha256'] = sha256(DUONG_DAN_MODEL.read_bytes()).hexdigest()
DUONG_DAN_METADATA.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n',
                              encoding='utf-8')
print('Model:', DUONG_DAN_MODEL, '\nMetadata:', DUONG_DAN_METADATA)


## 4. Nạp lại và smoke test

Thử dự đoán trên năm dòng **không kèm nhãn**, chỉ để xác nhận file nạp được và trả xác suất hợp lệ. Đây không phải Accuracy/AP mới. Validation và test lịch sử đã từng được xem nên không gọi kết quả đó là xác nhận độc lập.


In [5]:
loaded = joblib.load(DUONG_DAN_MODEL)
assert loaded['feature_columns'] == selected and loaded['cutoff_fraction'] == CUTOFF
sample = test_rows[selected].head(5)
probabilities = loaded['pipeline'].predict_proba(sample)[:, 1]
assert len(probabilities) == len(sample) and np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
print('Nạp thành công; 5 xác suất hợp lệ. Không đọc nhãn test.')
display(pd.DataFrame({'cutoff': [CUTOFF], 'số cột': [len(selected)],
                      'dòng fit': [len(fit_rows)], 'ngưỡng minh họa': [NGUONG]}))


Nạp thành công; 5 xác suất hợp lệ. Không đọc nhãn test.


,cutoff,số cột,dòng fit,ngưỡng minh họa
0,0.75,17,19712,0.5


## Giới hạn

Mask được chọn trên train-CV nhưng cấu hình và seed đã được xem trên validation; cần kỳ học mới để xác nhận ngoài mẫu. Được chọn không đồng nghĩa ảnh hưởng nhân quả. Xác suất chỉ hỗ trợ xem xét, không tự động quyết định học vụ. Chỉ nạp `joblib` từ nguồn tin cậy và môi trường tương thích.
